# 08 Batch Normalization 批量归一化

上一节讲正则化时提到 Batch Normalization，简称 BatchNorm 或 BN。

很多初学者会把 BatchNorm 只记成一个网络层，但这样记很空。我们这一节先讲清楚它为什么出现：**深层网络训练时，每一层输入分布会不断变化，BatchNorm 希望让这些中间数据更稳定。**

## 1. 为什么深层网络训练会不稳定

神经网络是一层接一层的复合函数：

$$
\mathbf{h}^{(1)}=f_1(\mathbf{x})
$$

$$
\mathbf{h}^{(2)}=f_2(\mathbf{h}^{(1)})
$$

$$
\mathbf{h}^{(3)}=f_3(\mathbf{h}^{(2)})
$$

第 $2$ 层的输入来自第 $1$ 层的输出，第 $3$ 层的输入来自第 $2$ 层的输出。

训练时，每一层的参数都在变。第 $1$ 层参数一变，它的输出分布就变；第 $2$ 层接收到的输入也跟着变。

所以后面的层会面对一个麻烦：自己正在学习，但输入数据的分布也一直在变。

## 2. 什么叫分布在变

这里的“分布”可以先理解成数据整体的形状，比如均值和方差。

如果一层收到的输入是：

$$
\mathbf{z}=[z_1,z_2,\dots,z_m]
$$

它的均值是：

$$
\mu=\frac{1}{m}\sum_{i=1}^{m}z_i
$$

方差是：

$$
\sigma^2=\frac{1}{m}\sum_{i=1}^{m}(z_i-\mu)^2
$$

如果训练过程中，某一层输入一会儿均值接近 $0$，一会儿均值变成 $10$；一会儿方差很小，一会儿方差很大，那么这层就会比较难学。

因为它面对的不是一个稳定任务，而是一个不断变化的任务。

## 3. BatchNorm 的核心想法

BatchNorm 的核心想法很直观：既然每一层输入分布容易乱，那就把它先整理一下。

整理成什么样？

先整理成均值为 $0$、方差为 $1$ 的数据：

$$
\hat{z}=\frac{z-\mu}{\sqrt{\sigma^2+\epsilon}}
$$

其中：

- $z$ 是某一层的中间输出。
- $\mu$ 是当前 batch 的均值。
- $\sigma^2$ 是当前 batch 的方差。
- $\epsilon$ 是一个很小的正数，防止除以 $0$。

这一步叫标准化。它让数据尺度更稳定。

## 4. 为什么叫 Batch Normalization

BatchNorm 里的 Batch 指的是 mini-batch。

训练时，我们不是拿全体训练集来算均值和方差，而是拿当前这个 batch 来算：

$$
\mu_B=\frac{1}{B}\sum_{i=1}^{B}z_i
$$

$$
\sigma_B^2=\frac{1}{B}\sum_{i=1}^{B}(z_i-\mu_B)^2
$$

然后标准化：

$$
\hat{z}_i=\frac{z_i-\mu_B}{\sqrt{\sigma_B^2+\epsilon}}
$$

所以 Batch Normalization 的名字很直白：用当前 batch 的统计量做归一化。

## 5. 只标准化会不会太死板

如果 BatchNorm 只做标准化，会有一个问题：它强行把数据变成均值 $0$、方差 $1$。

但神经网络有时候可能需要某一层输出不一定是这种分布。

所以 BatchNorm 在标准化之后，又加了两个可学习参数：

$$
y=\gamma\hat{z}+\beta
$$

其中：

- $\gamma$ 控制缩放。
- $\beta$ 控制平移。

也就是说，BatchNorm 先把数据整理干净，再允许模型自己决定要不要把它放大、缩小、平移。

## 6. 完整 BatchNorm 公式

对一个 batch 里的中间输出 $z_1,z_2,\dots,z_B$，BatchNorm 的计算过程是：

第一步，计算 batch 均值：

$$
\mu_B=\frac{1}{B}\sum_{i=1}^{B}z_i
$$

第二步，计算 batch 方差：

$$
\sigma_B^2=\frac{1}{B}\sum_{i=1}^{B}(z_i-\mu_B)^2
$$

第三步，标准化：

$$
\hat{z}_i=\frac{z_i-\mu_B}{\sqrt{\sigma_B^2+\epsilon}}
$$

第四步，缩放和平移：

$$
y_i=\gamma\hat{z}_i+\beta
$$

这就是 BatchNorm 的完整过程。

## 7. BatchNorm 到底放在哪里

在普通全连接网络中，常见位置是线性层之后、激活函数之前：

```text
Linear -> BatchNorm -> ReLU
```

用公式写就是：

$$
\mathbf{z}=\mathbf{W}\mathbf{x}+\mathbf{b}
$$

$$
\tilde{\mathbf{z}}=\operatorname{BN}(\mathbf{z})
$$

$$
\mathbf{h}=\operatorname{ReLU}(\tilde{\mathbf{z}})
$$

在 CNN 中，常见位置是卷积之后、激活函数之前：

```text
Conv -> BatchNorm -> ReLU
```

先记住这个常见顺序即可。真实模型里也会有不同变体，但入门阶段不要被变体扰乱。

## 8. BatchNorm 为什么能让训练更稳定

BatchNorm 的主要作用是稳定中间层输入的尺度。

如果没有 BatchNorm，某层输入可能越来越大或越来越小，激活函数容易进入不好的区域。

例如 Sigmoid 输入太大时会饱和：

$$
\sigma'(z)\approx 0
$$

Tanh 输入太大时也会饱和：

$$
\tanh'(z)\approx 0
$$

BatchNorm 把中间值拉回比较稳定的范围，可以缓解这种问题。

它的效果通常包括：

- 训练更稳定。
- 可以使用稍大的学习率。
- 对参数初始化没那么敏感。
- 深层网络更容易训练。

## 9. 训练阶段和推理阶段有什么区别

BatchNorm 在训练和推理时行为不同。

训练时，BatchNorm 使用当前 batch 的均值和方差：

$$
\mu_B,\quad \sigma_B^2
$$

但推理时，可能只有一个样本。如果只用一个样本算方差，统计量会非常不稳定。

所以推理时通常使用训练过程中积累下来的移动平均均值和移动平均方差：

$$
\mu_{running},\quad \sigma^2_{running}
$$

这就是为什么模型有训练模式和评估模式。BatchNorm 是最需要区分这两种模式的层之一。

## 10. BatchNorm 和 Dropout 的区别

BatchNorm 和 Dropout 都会让训练和推理阶段行为不同，但它们解决的问题不一样。

| 方法 | 主要目的 | 训练时做什么 | 推理时做什么 |
|---|---|---|---|
| BatchNorm | 稳定中间分布 | 用 batch 统计量归一化 | 用 running 统计量归一化 |
| Dropout | 缓解过拟合 | 随机关闭神经元 | 不关闭神经元 |

BatchNorm 的主业是让训练更顺。

Dropout 的主业是让模型不要过度依赖某些神经元。

## 11. BatchNorm 为什么可能有一点正则化效果

训练时，BatchNorm 使用当前 batch 的均值和方差。

不同 batch 的统计量会有轻微差异，所以同一个样本在不同 batch 中经过 BatchNorm 后，结果可能略有不同。

这种轻微扰动会让模型不那么容易死记训练样本，因此可能带来一点正则化效果。

但要记住：这不是 BatchNorm 的主要目的。

一句话：BatchNorm 主要为了稳定训练，顺带可能缓解一点过拟合。

## 12. BatchNorm 有什么局限

BatchNorm 依赖 batch 的统计量，所以 batch size 太小时，均值和方差估计可能不稳定。

如果 batch size 很小，比如只有 $1$ 或 $2$，当前 batch 可能不能代表整体数据分布。

这时 BatchNorm 的效果可能变差。

另外，在某些序列任务或在线推理场景中，batch 统计量也不一定方便使用。

这就是为什么后来还会有 LayerNorm、GroupNorm 等其他归一化方法。它们也是后面值得学的内容。

## 13. 和数据标准化有什么区别

数据标准化通常发生在输入层之前，比如把原始特征处理成均值 $0$、方差 $1$：

$$
x' = \frac{x-\mu}{\sigma}
$$

BatchNorm 发生在神经网络内部，处理的是中间层输出：

$$
z' = \operatorname{BN}(z)
$$

所以二者不是一回事：

| 方法 | 处理对象 | 发生位置 |
|---|---|---|
| 数据标准化 | 原始输入数据 | 模型之前 |
| BatchNorm | 中间层输出 | 模型内部 |

输入数据标准化是为了让模型一开始接收到更稳定的输入。

BatchNorm 是为了让模型内部每一层也尽量接收到更稳定的输入。

## 14. 本节总结

这一节的逻辑链是：

```text
深层网络中，前面层参数变化会影响后面层输入分布
-> 后面层面对不断变化的数据会更难训练
-> BatchNorm 用当前 batch 的均值和方差做标准化
-> 再用 gamma 和 beta 让模型自己学习合适的缩放和平移
-> 训练时用 batch 统计量
-> 推理时用 running 统计量
```

先记住三个公式：

batch 均值：

$$
\mu_B=\frac{1}{B}\sum_{i=1}^{B}z_i
$$

batch 标准化：

$$
\hat{z}_i=\frac{z_i-\mu_B}{\sqrt{\sigma_B^2+\epsilon}}
$$

缩放和平移：

$$
y_i=\gamma\hat{z}_i+\beta
$$

下一节可以继续讲一个真正的完整 MLP 结构：输入层、隐藏层、输出层、参数形状、二分类和多分类输出怎么设计。